In [ ]:
import pandas as pd
import matplotlib as plt
import json
from datetime import datetime, timedelta
df_fall_26 = pd.read_csv('output/fall_26.csv')
df_spring_26 = pd.read_csv('output/spring_26.csv')
df_fall_25 = pd.read_csv('output/fall_25_new_scrape.csv')

desired_codes = ['SEAS', 'SAS'] 

df_fall_26
df_spring_26
df_fall_25
#group class times

def group_class(df, term):
    # drop rows with missing days/time
    df = df.dropna(subset=['days'])
    df['start_time'] = pd.to_datetime(df['start_time'], format='%H:%M')
    df['end_time'] = pd.to_datetime(df['end_time'], format='%H:%M')

    df['days'] = df['days'].str.split().str[0]


    day_map = {
        'M': 'Monday',
        'T': 'Tuesday',
        'W': 'Wednesday',
        'R': 'Thursday',
        'F': 'Friday'
    }
    df['days'] = df['days'].fillna('').astype(str)

    df['days'] = df['days'].apply(
        lambda x: [day_map.get(d) for d in x if d in day_map]
    )
    df = df.explode('days').reset_index(drop=True)

    time_bins = pd.date_range("08:00", "23:00", freq="30min").time

    rows = []

    for _, row in df.iterrows():
        day = row['days']
        start = row['start_time'].time()
        end = row['end_time'].time()

        for t in time_bins:
            # convert time → minutes since midnight
            t_minutes = t.hour * 60 + t.minute
            bin_start_minutes = t_minutes
            bin_end_minutes = t_minutes + 30

            start_minutes = start.hour * 60 + start.minute
            end_minutes = end.hour * 60 + end.minute

            # overlap condition
            if start_minutes < bin_end_minutes and end_minutes > bin_start_minutes:
                rows.append({
                    'Day': day,
                    'Time': t
                })

    binned_df = pd.DataFrame(rows)
    # binned_df.to_csv(f"filtered_{term}.csv")
    result = (
        binned_df
        .groupby(['Day', 'Time'])
        .size()
        .reset_index(name='Share')
    )

    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

    result['Day'] = pd.Categorical(result['Day'], categories=day_order, ordered=True)
    result['Time'] = pd.to_datetime(result['Time'].astype(str)).dt.time

    result = result.sort_values(['Day', 'Time'])

    result.to_csv(f'{term}.csv', index=False)

    return result

# desired_type = ['-REC', '-DISC', 'recitation', 'discussion', 'lab', '- DISC', '- REC', 'DISC -', 'DISC-', 'REC -']
# pattern = '|'.join(desired_type)

desired_type = [r'\bLAB\b', r'\bREC\b', r'\bDISC\b', r'\bDISCUSSION\b',
                r'\bRECITATION\b', r'\bLABORATORY\b', '-REC', '-DISC', '-LAB',
                'independent study']
pattern = '|'.join(desired_type)

df_filtered = df[
    df['school_code'].isin(desired_codes) &
    ~df['course_name'].str.contains(pattern, case=False, na=False)
]


# display(fall_filtered_df.shape[0])

# display(fall_filtered_df)

# fall_filtered_df.to_csv(f'testing.csv', index=False)


In [37]:

# grad_pattern = r'.{4}G'
# remove = df['course.course_identifier'].str.contains(grad_pattern, regex=True, na=False)
# df_undergrad = df[~remove]
# df_undergrad


fall_filtered_df = df_fall_26[df_fall_26['school_code'].isin(desired_codes)]
display(fall_filtered_df.shape[0])
# fall_filtered_df.to_csv(f'testing.csv', index=False)
fall_filtered_df = fall_filtered_df[~fall_filtered_df['course_name'].str.contains(pattern, case=False, na=False)]
fall_filtered_df.to_csv(f'testing.csv', index=False)
display(fall_filtered_df.shape[0])
# display(fall_filtered_df)

fall_filtered_df = fall_filtered_df.dropna(subset=['days'])
fall_filtered_df = fall_filtered_df.dropna(subset=['end_time'])
fall_filtered_df = fall_filtered_df.dropna(subset=['start_time'])
display(fall_filtered_df.shape[0])
fall_filtered_df.to_csv(f'testing_fall_26.csv', index=False)
# display(fall_filtered_df.shape[0])
display(group_class(fall_filtered_df, 'fall 26 new'))

2630

2223

1582

/var/folders/zl/l01js9_527d_nstkmhybfn0c0000gn/T/ipykernel_54465/1264421810.py:76: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  result['Time'] = pd.to_datetime(result['Time'].astype(str)).dt.time


,Day,Time,Share
27,Monday,08:00:00,11
28,Monday,08:30:00,49
29,Monday,09:00:00,57
30,Monday,09:30:00,57
31,Monday,10:00:00,114
...,...,...,...
22,Friday,19:00:00,3
23,Friday,19:30:00,4
24,Friday,20:00:00,2
25,Friday,20:30:00,2


In [38]:

spring_filtered_df = df_spring_26[df_spring_26['school_code'].isin(desired_codes)]
# display(fall_filtered_df.shape[0])
spring_filtered_df = spring_filtered_df[~spring_filtered_df['course_name'].str.contains(pattern, case=False, na=False)]
spring_filtered_df = spring_filtered_df.dropna(subset=['days'])
spring_filtered_df = spring_filtered_df.dropna(subset=['end_time'])
spring_filtered_df = spring_filtered_df.dropna(subset=['start_time'])
display(spring_filtered_df.shape[0])
display(group_class(spring_filtered_df, 'spring 26 new'))



fall_25_filtered_df = df_fall_25[df_fall_25['school_code'].isin(desired_codes)]
fall_25_filtered_df = fall_25_filtered_df[~fall_25_filtered_df['course_name'].str.contains(pattern, case=False, na=False)]
fall_25_filtered_df = fall_25_filtered_df.dropna(subset=['days'])
fall_25_filtered_df = fall_25_filtered_df.dropna(subset=['end_time'])
fall_25_filtered_df = fall_25_filtered_df.dropna(subset=['start_time'])
display(fall_25_filtered_df.shape[0])
display(group_class(fall_25_filtered_df, 'fall 25 new'))


1769

/var/folders/zl/l01js9_527d_nstkmhybfn0c0000gn/T/ipykernel_54465/1264421810.py:76: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  result['Time'] = pd.to_datetime(result['Time'].astype(str)).dt.time


,Day,Time,Share
22,Monday,08:00:00,10
23,Monday,08:30:00,54
24,Monday,09:00:00,60
25,Monday,09:30:00,61
26,Monday,10:00:00,142
...,...,...,...
17,Friday,16:30:00,13
18,Friday,17:00:00,9
19,Friday,17:30:00,6
20,Friday,18:00:00,6


1813

/var/folders/zl/l01js9_527d_nstkmhybfn0c0000gn/T/ipykernel_54465/1264421810.py:76: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  result['Time'] = pd.to_datetime(result['Time'].astype(str)).dt.time


,Day,Time,Share
28,Monday,08:00:00,11
29,Monday,08:30:00,56
30,Monday,09:00:00,66
31,Monday,09:30:00,66
32,Monday,10:00:00,132
...,...,...,...
23,Friday,19:30:00,6
24,Friday,20:00:00,2
25,Friday,20:30:00,2
26,Friday,21:00:00,2
